In [1]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import random
from scipy.io import loadmat
from scipy.stats import spearmanrho
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])


In [16]:
from scipy import stats
def permutation_test_corr(
    x: np.ndarray,
    y: np.ndarray,
    n_perm: int = 10000,
    corr_type: str = "pearson",  # "pearson" or "spearman"
    test_type: str = "two",   # "two", "upper", "lower"
) -> tuple[float, float, np.ndarray]:

    x, y = np.asarray(x), np.asarray(y)

    if corr_type == "pearson":
        corr_fn = lambda a, b: np.corrcoef(a, b)[0, 1]
    elif corr_type == "spearman":
        corr_fn = lambda a, b: stats.spearmanr(a, b).statistic
    else:
        raise ValueError(f"corr_type must be 'pearson' or 'spearman', got '{corr_type}'")

    obs = corr_fn(x, y)

    null_dist = np.zeros(n_perm)
    for i in range(n_perm):
        null_dist[i] = corr_fn(x, np.random.permutation(y))

    if test_type == "two":
        p = np.mean(np.abs(null_dist) >= np.abs(obs))
    elif test_type == "upper":
        p = np.mean(null_dist >= obs)
    elif test_type == "lower":
        p = np.mean(null_dist <= obs)
    else:
        raise ValueError(f"test_type must be 'two', 'upper', or 'lower', got '{test_type}'")

    return obs, p, null_dist

In [17]:
rhos = np.array([ 0.675,  0.903,  0.064,  0.935,  0.589,  0.494, -0.165,  0.367, -0.17 ,  0.899])
layers_fit = np.array([0.00416, 0.00512, 0.00329, 0.00385, 0.00373, 0.00525, 0.00186, 0.00367, 0.00137, 0.00786])

In [20]:
obs, p, null = permutation_test_corr(rhos, layers_fit, n_perm=10000, test_type="upper", corr_type="pearson")
print(obs, p)

0.7822703605638607 0.0015
